# Model comparison

## 1. Importing libraries

In [9]:
import numpy as np
import pandas as pd
import arviz as az
import xarray as xr
import gc
import unicodedata
import re
from collections import Counter
import ast

## 2. Defining helper functions

In [10]:
def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def a_loglik(idata):
    """True if the log_likelihood group exists.
    ArviZ 1.x : idata.groups contains paths ('/posterior', '/log_likelihood'),
    so  "log_likelihood" in idata.groups  is always False."""
    return any(g.strip("/") == "log_likelihood" for g in idata.groups)


def _g(res, *noms):
    """Reads an attribute from the az.loo result depending on the ArviZ version
    (1.x: elpd, p, elpd_i; 0.x: elpd_loo, p_loo, loo_i)."""
    for n in noms:
        if hasattr(res, n):
            return getattr(res, n)
    raise AttributeError(noms)


def _extraire(idata, df_bayes, predicteurs_modele, col_voie):
    """Extracts everything needed from the posterior to recompute eta_i."""
    post = idata.posterior
    n_chains, n_draws = post.sizes["chain"], post.sizes["draw"]
    seuils = post["threshold"].values.reshape(n_chains * n_draws, -1)   # (sims, K-1)
    betas = np.stack([post[p].values.reshape(-1)
                      for p in predicteurs_modele])                     # (P, sims)
    nom_u = f"1|{col_voie}"
    if nom_u in post:
        u = post[nom_u].values.reshape(n_chains * n_draws, -1)          # (sims, n_voies)
        # actual route dimension name (do not assume "<name>_dim")
        dim_u = [d for d in post[nom_u].dims if d not in ("chain", "draw")][0]
        if dim_u in post[nom_u].coords:
            coords = np.asarray(post[nom_u].coords[dim_u].values).astype(str)
        else:                                   # fallback: category order in df_bayes
            coords = np.asarray(df_bayes[col_voie].cat.categories).astype(str)
        pos = {v: i for i, v in enumerate(coords)}
        route_codes = df_bayes[col_voie].astype(str).map(pos).values
        assert route_codes.min() >= 0, "Some df_bayes routes are missing from the offsets."
    else:
        u, route_codes = None, None        # model without a random intercept (A)
    X = df_bayes[predicteurs_modele].astype(float).values               # (n_obs, P)
    y = df_bayes["classe_cotation"].cat.codes.values                    # 0..K-1
    return post, n_chains, n_draws, seuils, betas, u, route_codes, X, y


def _calcule_plage(X, y, seuils, betas, u, route_codes, d0, d1, signe):
    """Log-likelihood of draws [d0, d1), vectorized. Returns (d1-d0, n_obs)."""
    n_obs, K = X.shape[0], seuils.shape[1] + 1
    eta = X @ betas[:, d0:d1]                       # (n_obs, chunk)
    if u is not None:
        eta = eta + u[d0:d1, route_codes].T         # (chunk, n_obs).T -> (n_obs, chunk)
    th = seuils[d0:d1]                              # (chunk, K-1)
    p_prev = np.zeros_like(eta)                     # P(Y <= -1) = 0
    ll = np.empty((d1 - d0, n_obs), dtype=np.float64)
    for k in range(K - 1):
        p_k = _sigmoid(signe * (eta - th[:, k][None, :]))     # P(Y <= k)
        logp = np.log(np.clip(p_k - p_prev, 1e-300, None))   # log P(Y = k)
        ll[:, y == k] = logp[y == k, :].T
        p_prev = p_k
    logp = np.log(np.clip(1.0 - p_prev, 1e-300, None))       # log P(Y = K-1)
    ll[:, y == K - 1] = logp[y == K - 1, :].T
    return ll


def valide_convention(idata, df_bayes, predicteurs_modele, col_voie="document_id",
                      n_sims=200):
    """
    Compares the NumPy calculation with the integrated log-likelihood over the first n_sims
    draws, for the two possible sign conventions:
      +1 : P(Y <= k) = sigmoid(eta - seuil_k)
      -1 : P(Y <= k) = sigmoid(seuil_k - eta)
    Returns (signe_validé, dict of maximum discrepancies).
    """
    assert a_loglik(idata), \
        "This model has no stored log_likelihood: nothing to validate."
    (post, n_chains, n_draws, seuils, betas, u, route_codes, X, y) = _extraire(
        idata, df_bayes, predicteurs_modele, col_voie)
    var_nom = list(idata.log_likelihood.data_vars)[0]
    stocke = idata.log_likelihood[var_nom].values.reshape(n_chains * n_draws, -1)
    n_test = min(n_sims, stocke.shape[0])
    ecarts = {}
    for signe in (+1, -1):
        ll = _calcule_plage(X, y, seuils, betas, u, route_codes,
                           0, n_test, signe)
        ecarts[signe] = float(np.max(np.abs(ll - stocke[:n_test])))
    bon = min(ecarts, key=ecarts.get)
    if ecarts[bon] < 1e-4:
        print(f"Sign convention {bon:+d} VALIDATED — max deviation vs. integrated log-likelihood: "
              f"{ecarts[bon]:.2e}")
    else:
        print(f"⚠️ No sign convention matches: maximum discrepancies = {ecarts}")
    return bon, ecarts


def calcule_loglik(idata, df_bayes, predicteurs_modele, col_voie="document_id",
                   taille_chunk=100, signe=None, chemin_sortie=None):
    """
    Calculates the log-likelihood and attaches it to idata as the log_likelihood group.

    signe: sign convention validated by valide_convention (+1 by default if absent — validate on C or D first!).
    chemin_sortie: if provided, saves the completed idata (DO NOT overwrite
    the original — keep the raw .nc file intact).
    """
    (post, n_chains, n_draws, seuils, betas, u, route_codes, X, y) = _extraire(
        idata, df_bayes, predicteurs_modele, col_voie)
    n_obs, n_sims = X.shape[0], n_chains * n_draws
    if signe is None:
        if a_loglik(idata):
            signe, _ = valide_convention(idata, df_bayes, predicteurs_modele, col_voie)
        else:
            signe = +1
            print("⚠️ No log_likelihood stored: defaulting to sign +1. "
                  "Validate on C or D first.")
    print(f"Calculating log-likelihood: {n_sims} draws × {n_obs} observations "
          f"(sign convention {signe:+d}, chunks of {taille_chunk})...")
    ll = np.empty((n_sims, n_obs), dtype=np.float32)   # use float32 from the outset: half the memory
    for d0 in range(0, n_sims, taille_chunk):
        d1 = min(d0 + taille_chunk, n_sims)
        ll[d0:d1] = _calcule_plage(X, y, seuils, betas, u, route_codes,
                                   d0, d1, signe)
    # Assemble the log_likelihood group (float32: half the size)
    da = xr.DataArray(
        ll.reshape(n_chains, n_draws, n_obs).astype(np.float32),
        dims=("chain", "draw", "obs"),
        coords={"chain": np.arange(n_chains), "draw": np.arange(n_draws)},
        name="classe_cotation",
    )
    if a_loglik(idata):
        del idata["log_likelihood"]
    idata["log_likelihood"] = xr.DataTree(dataset=xr.Dataset({"classe_cotation": da}))
    print(f"OK — log-likelihood added (mean log-likelihood: {ll.mean():.2f}).")
    if chemin_sortie:
        idata.to_netcdf(chemin_sortie)
        print(f"Completed idata saved: {chemin_sortie}")
    return idata

## Comparison

In [11]:
# ══════════════════════════════════════════════════════════
# Cell 2: loading and basic preparation (SHARED)
# ══════════════════════════════════════════════════════════
ordre_cotations = [
    "1a", "1b", "1b+", "1c+",
    "2a", "2a+", "2b", "2b+", "2c", "2c+",
    "3a", "3a+", "3b", "3b+", "3c", "3c+",
    "4a", "4a+", "4b", "4b+", "4c", "4c+",
    "5a", "5a+", "5b", "5b+", "5c", "5c+",
    "6a", "6a+", "6b", "6b+", "6c", "6c+",
    "7a", "7a+", "7b", "7b+", "7c", "7c+",
    "8a", "8a+", "8b", "8b+", "8c", "8c+", "9a+",
]

df = pd.read_csv("annotation_finale_ponderee.csv")
df["difficulty"] = pd.Categorical(df["difficulty"], categories=ordre_cotations, ordered=True)

elements_frequents = ["dalle", "fissure", "dièdre", "traversée", "surplomb",
                      "dévers", "cheminée", "vire", "arête", "pilier",
                      "terrasse", "éperon"]
elements_intermediaires = ["écailles", "trou", "couloir", "cannelure", "réglette",
                           "goutte_eau", "crochet", "bac", "plat",
                           "petite_prise"]

# Keep ALL columns (termes_ordonnes, pond_* will be
# needed for C and D) — selection is based on the predictor list.
df_model = df.copy()
for e in elements_frequents:
    df_model[f"nb_{e}"] = pd.to_numeric(df_model[f"nb_{e}"]).clip(upper=3)
for e in elements_intermediaires:
    df_model[f"pres_{e}"] = (pd.to_numeric(df_model[f"nb_{e}"]) > 0).astype(int)
print(f"{len(df_model)} rows loaded")

42861 rows loaded


C:\Users\lunag\AppData\Local\Temp\ipykernel_36620\1250706698.py:15: DtypeWarning: Columns (31) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("annotation_finale_ponderee.csv")


In [12]:
# ══════════════════════════════════════════════════════════
# Cell 3: tools + shared preparation function
# ══════════════════════════════════════════════════════════

def nettoyer_nom(nom):
    s = unicodedata.normalize("NFKD", nom).encode("ascii", "ignore").decode()
    return re.sub(r"\W+", "_", s).strip("_")     # ← "_" not "*" (bug fixed)

ORDRE = [f"{n}{l}{p}" for n in range(1, 10) for l in "abc" for p in ["", "+"]]
RANG = {g: i for i, g in enumerate(ORDRE)}

def preparer_modele(df_model, predicteurs, col_voie="document_id",
                    cotation_min="4c+", cotation_max="7a+",
                    separer=False, seuil_fusion=400):
    """Reproduces your preparation exactly: classes, (decomposition),
    merging of rare categories, standardization, ordinal response."""
    rang = df_model["difficulty"].astype("string").str.strip().str.lower() \
                .map(RANG).astype(float)
    bas, haut = RANG[cotation_min], RANG[cotation_max]
    data = pd.DataFrame({
        "classe_cotation": (rang.clip(lower=bas, upper=haut) - bas + 1),
        col_voie: df_model[col_voie],
    })
    noms_propres = {p: nettoyer_nom(p) for p in predicteurs}
    for p, propre in noms_propres.items():
        data[propre] = pd.to_numeric(df_model[p], errors="coerce")
    preds = list(noms_propres.values())
    data = data.dropna(subset=["classe_cotation"] + preds).copy()
    data["classe_cotation"] = data["classe_cotation"].astype(int)

    if separer:                                   # model A only
        nouvelles = []
        for p in preds:
            moyenne = data.groupby(col_voie)[p].transform("mean")
            data[p + "_ecart"] = data[p] - moyenne
            data[p + "_moy_voie"] = moyenne
            nouvelles += [p + "_ecart", p + "_moy_voie"]
        preds = nouvelles

    effectifs = data["classe_cotation"].value_counts().sort_index()
    mapping = {c: (c - 1 if effectifs[c] < seuil_fusion and (c - 1) in effectifs.index
                   else c) for c in effectifs.index}
    data["classe_cotation"] = data["classe_cotation"].map(mapping)

    for p in preds:                               # standardization + winsorization
        x = data[p].astype(float)
        data[p] = ((x - x.mean()) / x.std()).clip(-10, 10)

    categories = sorted(data["classe_cotation"].unique())
    data["classe_cotation"] = pd.Categorical(data["classe_cotation"],
                                             categories=categories, ordered=True)
    data[col_voie] = data[col_voie].astype("category")
    print(f"  → {len(data)} rows, {data[col_voie].nunique()} routes, "
          f"{data['classe_cotation'].nunique()} classes")
    return data, preds

In [13]:
# ══════════════════════════════════════════════════════════
# Cell 4: the four preparations → PREPS
# ══════════════════════════════════════════════════════════
PREDICTEURS_B = ['nb_dalle', 'nb_fissure', 'nb_dièdre', 'nb_traversée',
                 'nb_surplomb', 'nb_dévers', 'nb_cheminée', 'nb_vire',
                 'nb_arête', 'nb_pilier', 'nb_terrasse', 'nb_éperon',
                 'pres_écailles', 'pres_trou', 'pres_couloir', 'pres_cannelure',
                 'pres_réglette', 'pres_goutte_eau', 'pres_crochet', 'pres_bac',
                 'pres_plat', 'pres_petite_prise', 'log_nb_terms_description']

PREPS = {}
print("Model B:");  PREPS["B"] = preparer_modele(df_model, PREDICTEURS_B)
print("Model A:");  PREPS["A"] = preparer_modele(df_model, PREDICTEURS_B, separer=True)

Model B:
  → 42861 rows, 9299 routes, 14 classes
Model A:
  → 42861 rows, 9299 routes, 14 classes


In [14]:
# ── Cell 4bis: Model C — transition extraction ──

SEQ_COL = "termes_ordonnes"
SEUIL_TRANSITION = 400

def vers_liste(s):
    if isinstance(s, list):
        return s
    if pd.isna(s):
        return []
    try:
        v = ast.literal_eval(s)
        return v if isinstance(v, list) else [s]
    except (ValueError, SyntaxError):
        return [s]

seqs = df_model[SEQ_COL].apply(vers_liste)
freq = Counter()
for s in seqs:
    for a, b in zip(s, s[1:]):
        freq[(a, b)] += 1
retenues = sorted([t for t, n in freq.items() if n >= SEUIL_TRANSITION],
                  key=lambda t: -freq[t])
print(f"{len(retenues)} transitions retained")     # ← should display 24

noms_trans = []
for (a, b) in retenues:
    compte = Counter()
    for idx, s in zip(seqs.index, seqs):
        for x, y in zip(s, s[1:]):
            if (x, y) == (a, b):
                compte[idx] += 1
    nom = nettoyer_nom(f"trans_{a}_vers_{b}")
    df_model[nom] = pd.Series(compte).reindex(df_model.index,
                                              fill_value=0).astype(float)
    noms_trans.append(nom)

PREDICTEURS_C = PREDICTEURS_B + noms_trans
print("Model C:")
PREPS["C"] = preparer_modele(df_model, PREDICTEURS_C)

27 transitions retained
Model C:
  → 42861 rows, 9299 routes, 14 classes


In [15]:
# ── Cell 4ter: Model D — weights ──
POND_RETENUES = ["dalle_ponderation", "fissure_ponderation", "dièdre_ponderation",
                 "traversée_ponderation", "surplomb_ponderation", "vire_ponderation",
                 "dévers_ponderation", "terrasse_ponderation"]
PREDICTEURS_D = PREDICTEURS_B + POND_RETENUES
print("Model D:")
PREPS["D"] = preparer_modele(df_model, PREDICTEURS_D)

Model D:


  → 42861 rows, 9299 routes, 14 classes


In [16]:
# ══════════════════════════════════════════════════════════
# Cell 5 (replaces the old cells 5 and 6): NumPy log-likelihood + LOO for the four models
# ══════════════════════════════════════════════════════════
def signe_par_vraisemblance(idata, df_b, preds, n_tirages=50):
    """Chooses the sign convention WITHOUT a stored log-likelihood: the correct one is the one
    that makes the observed data substantially more likely under the posterior distribution."""
    (post, nc, nd, seuils, betas, u, rc, X, y) = _extraire(idata, df_b, preds, "document_id")
    idx = np.linspace(0, nc * nd - 1, n_tirages).astype(int)      # draws distributed across chains
    moyennes = {}
    for s in (+1, -1):
        moyennes[s] = float(np.mean([_calcule_plage(X, y, seuils, betas, u, rc, d, d + 1, s).sum()
                                     for d in idx]))
    bon = max(moyennes, key=moyennes.get)
    autre = -bon
    print(f"  mean log-likelihood: +1 → {moyennes[+1]:,.0f} | −1 → {moyennes[-1]:,.0f}"
          f"  ⇒ sign convention {bon:+d}")
    assert moyennes[bon] - moyennes[autre] > 100, "Gap too small: sign convention is not identifiable."
    return bon


# Values from the initial runs (with Bambi log-likelihood): a simple consistency check
REFERENCE = {"A": -100209.2, "C": -86731.6, "D": -86510.4}

RESULTATS, SIGNE = {}, None
for nom in ["A", "B", "C", "D"]:
    print(f"── Model {nom} ──")
    df_b, preds = PREPS[nom]
    idata = az.from_netcdf(f"idata_{nom}.nc")
    if SIGNE is None:
        SIGNE = signe_par_vraisemblance(idata, df_b, preds)
    idata = calcule_loglik(idata, df_b, preds, signe=SIGNE)       # no saving (about 1 GB per model)
    loo = az.loo(idata, pointwise=True)
    # Keep only what is needed: the full result contains the PSIS weights (about 2 GB per model)
    RESULTATS[nom] = {
        "elpd": float(_g(loo, "elpd", "elpd_loo")),
        "se": float(loo.se),
        "p": float(_g(loo, "p", "p_loo")),
        "elpd_i": np.asarray(_g(loo, "elpd_i", "loo_i").values).copy(),
        "pareto_k": np.asarray(loo.pareto_k.values).copy(),
    }
    msg = f"  {nom} → elpd_loo = {RESULTATS[nom]['elpd']:,.1f}"
    if nom in REFERENCE:
        msg += f"  (initial run: {REFERENCE[nom]:,.1f} ; difference {RESULTATS[nom]['elpd'] - REFERENCE[nom]:+.1f})"
    print(msg)
    del idata, loo
    gc.collect()
print("Sign convention shared by all four models:", SIGNE)


── Modèle A ──
  mean log-likelihood: +1 → -24,591,844 | −1 → -82,206  ⇒ sign convention -1
Calculating log-likelihood: 6000 draws × 42861 observations (sign convention -1, chunks of 100)...
OK — log-likelihood added (mean log-likelihood: -1.92).


c:\Python313\Lib\site-packages\arviz_stats\loo\loo_helper.py:1169: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


  A → elpd_loo = -86,551.5  (initial run: -100,209.2 ; difference +13657.7)
── Modèle B ──
Calculating log-likelihood: 6000 draws × 42861 observations (sign convention -1, chunks of 100)...
OK — log-likelihood added (mean log-likelihood: -1.92).


c:\Python313\Lib\site-packages\arviz_stats\loo\loo_helper.py:1169: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


  B → elpd_loo = -86,685.4
── Modèle C ──
Calculating log-likelihood: 6000 draws × 42861 observations (sign convention -1, chunks of 100)...
OK — log-likelihood added (mean log-likelihood: -1.92).


c:\Python313\Lib\site-packages\arviz_stats\loo\loo_helper.py:1169: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


  C → elpd_loo = -86,677.8  (initial run: -86,731.6 ; difference +53.8)
── Modèle D ──
Calculating log-likelihood: 6000 draws × 42861 observations (sign convention -1, chunks of 100)...
OK — log-likelihood added (mean log-likelihood: -1.91).


c:\Python313\Lib\site-packages\arviz_stats\loo\loo_helper.py:1169: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


  D → elpd_loo = -86,482.9  (initial run: -86,510.4 ; difference +27.5)
Sign convention shared by all four models: -1


In [17]:
# ══════════════════════════════════════════════════════════
# Cell 6: comparison tables
# ══════════════════════════════════════════════════════════
lignes = {n: {"elpd_loo": round(r["elpd"], 1),
              "se": round(r["se"], 1),
              "p_loo": round(r["p"], 1),
              "part_k>0.7": round(float((r["pareto_k"] > 0.7).mean()), 3)}
          for n, r in RESULTATS.items()}
print(pd.DataFrame(lignes).T.sort_values("elpd_loo", ascending=False))

loo_i = {n: r["elpd_i"] for n, r in RESULTATS.items()}
meilleur = max(loo_i, key=lambda k: loo_i[k].sum())
comp = []
for nom, li in loo_i.items():
    diff = loo_i[meilleur] - li
    comp.append({"model": nom,
                 "elpd_loo": round(li.sum(), 1),
                 "diff_elpd": round(diff.sum(), 1),
                 "se_diff": round(float(np.sqrt(len(li) * diff.var(ddof=1))), 1)
                 if nom != meilleur else 0.0})
print(f"\nReference model: {meilleur}")
print(pd.DataFrame(comp).sort_values("diff_elpd").to_string(index=False))


   elpd_loo     se   p_loo  part_k>0.7
D  -86482.9  179.4  7692.3       0.004
A  -86551.5  179.4  7522.7       0.002
C  -86677.8  179.1  7709.4       0.003
B  -86685.4  179.0  7681.1       0.004

Reference model: D
model  elpd_loo  diff_elpd  se_diff
     D  -86482.9        0.0      0.0
     A  -86551.5       68.7     32.3
     C  -86677.8      195.0     24.3
     B  -86685.4      202.5     22.2


In [18]:
# ============================================================
# 7. GLOBAL McKELVEY-ZAVOINA R²
# ============================================================

def calcule_r2_mz_global(
    idata,
    df_bayes,
    predicteurs_modele,
    col_voie="document_id",
    taille_chunk=50
):
    """
    Calculates the McKelvey-Zavoina R² for each posterior draw.

    Includes:
    - fixed effects of the predictors;
    - route-specific random intercepts.

    Returns a table containing one R² per draw.
    """

    (
        post, n_chains, n_draws,
        seuils, betas, u, route_codes, X, y
    ) = _extraire(
        idata, df_bayes, predicteurs_modele, col_voie
    )

    # Required check: route effects must be included
    if u is None:
        raise ValueError(
            f"The random intercepts for {col_voie} "
            "were not found in the posterior distribution."
        )

    n_sims = n_chains * n_draws

    r2_draws = np.empty(n_sims)

    variance_residuelle = np.pi**2 / 3

    # Process in chunks to limit memory use
    for d0 in range(0, n_sims, taille_chunk):

        d1 = min(d0 + taille_chunk, n_sims)

        # Contribution of the explanatory variables
        eta = X @ betas[:, d0:d1]

        # Add route-specific effects
        eta += u[d0:d1, route_codes].T

        # Variance of the linear predictor across observations
        variance_eta = np.var(eta, axis=0, ddof=0)

        # McKelvey-Zavoina R²
        r2_draws[d0:d1] = (
            variance_eta /
            (variance_eta + variance_residuelle)
        )

    return r2_draws


# ============================================================
# APPLY TO THE FOUR MODELS
# ============================================================

R2_DRAWS = {}
R2_RESULTATS = {}

for nom in ["A", "B", "C", "D"]:

    print(f"\nCalculating R² for model {nom}...")

    df_b, preds = PREPS[nom]

    idata = az.from_netcdf(f"idata_{nom}.nc")

    r2 = calcule_r2_mz_global(
        idata=idata,
        df_bayes=df_b,
        predicteurs_modele=preds,
        col_voie="document_id"
    )

    R2_DRAWS[nom] = r2

    R2_RESULTATS[nom] = {
        "R2_median": np.median(r2),
        "R2_moyenne": np.mean(r2),
        "IC95_bas": np.quantile(r2, 0.025),
        "IC95_haut": np.quantile(r2, 0.975)
    }

    print(
        f"Model {nom}: "
        f"R² = {np.median(r2):.3f} "
        f"[{np.quantile(r2, 0.025):.3f} ; "
        f"{np.quantile(r2, 0.975):.3f}]"
    )

    del idata
    gc.collect()


# ============================================================
# COMPARISON TABLE
# ============================================================

tableau_r2 = pd.DataFrame(R2_RESULTATS).T

print("\nComparison of McKelvey-Zavoina R² values:")
display(tableau_r2.round(4))

tableau_r2.to_csv(
    "comparaison_R2_McKelvey_Zavoina.csv"
)



Calculating R² for model A...
Model A: R² = 0.653 [0.648 ; 0.659]

Calculating R² for model B...
Model B: R² = 0.653 [0.647 ; 0.659]

Calculating R² for model C...
Model C: R² = 0.654 [0.647 ; 0.659]

Calculating R² for model D...
Model D: R² = 0.656 [0.650 ; 0.662]

Comparison of McKelvey-Zavoina R² values:


,R2_median,R2_moyenne,IC95_bas,IC95_haut
A,0.6534,0.6534,0.6476,0.6592
B,0.6528,0.6528,0.6468,0.6587
C,0.6535,0.6535,0.6474,0.6594
D,0.6563,0.6563,0.6505,0.6619


In [19]:
# ============================================================
# 8. MCKELVEY-ZAVOINA R²: MARGINAL AND CONDITIONAL
# ============================================================

def calcule_r2_mz(
    idata,
    df_bayes,
    predicteurs_modele,
    col_voie="document_id",
    inclut_voie=True,
    taille_chunk=50
):
    """
    Calculates the McKelvey-Zavoina R² for each posterior draw.

    - inclut_voie=True  : fixed effects + random intercepts
                          for routes  ->  CONDITIONAL R²
    - inclut_voie=False : fixed effects only
                          ->  MARGINAL R²
    """

    (
        post, n_chains, n_draws,
        seuils, betas, u, route_codes, X, y
    ) = _extraire(
        idata, df_bayes, predicteurs_modele, col_voie
    )

    if inclut_voie and u is None:
        raise ValueError(
            f"The random intercepts for {col_voie} "
            "were not found in the posterior distribution."
        )

    n_sims = n_chains * n_draws

    r2_draws = np.empty(n_sims)

    variance_residuelle = np.pi**2 / 3

    for d0 in range(0, n_sims, taille_chunk):

        d1 = min(d0 + taille_chunk, n_sims)

        # Contribution of the explanatory variables
        eta = X @ betas[:, d0:d1]

        # Add route-specific effects (conditional model only)
        if inclut_voie:
            eta += u[d0:d1, route_codes].T

        variance_eta = np.var(eta, axis=0, ddof=0)

        r2_draws[d0:d1] = (
            variance_eta /
            (variance_eta + variance_residuelle)
        )

    return r2_draws


# ============================================================
# APPLY TO THE FOUR MODELS — BOTH VERSIONS
# ============================================================

R2_MZ = {}

for nom in ["A", "B", "C", "D"]:

    print(f"\nCalculating R² values for model {nom}...")

    df_b, preds = PREPS[nom]

    idata = az.from_netcdf(f"idata_{nom}.nc")

    r2_marginal    = calcule_r2_mz(
        idata, df_b, preds, inclut_voie=False
    )
    r2_conditionnel = calcule_r2_mz(
        idata, df_b, preds, inclut_voie=True
    )

    R2_MZ[nom] = {
        "R2_marginal_median":    np.median(r2_marginal),
        "R2_marginal_IC_bas":    np.quantile(r2_marginal, 0.025),
        "R2_marginal_IC_haut":   np.quantile(r2_marginal, 0.975),
        "R2_conditionnel_median":  np.median(r2_conditionnel),
        "R2_conditionnel_IC_bas":  np.quantile(r2_conditionnel, 0.025),
        "R2_conditionnel_IC_haut": np.quantile(r2_conditionnel, 0.975),
    }

    print(
        f"Model {nom}: "
        f"Marginal R² = {np.median(r2_marginal):.3f} "
        f"[{np.quantile(r2_marginal, 0.025):.3f} ; "
        f"{np.quantile(r2_marginal, 0.975):.3f}] | "
        f"Conditional R² = {np.median(r2_conditionnel):.3f} "
        f"[{np.quantile(r2_conditionnel, 0.025):.3f} ; "
        f"{np.quantile(r2_conditionnel, 0.975):.3f}]"
    )

    del idata
    gc.collect()


# ============================================================
# COMPARISON TABLE
# ============================================================

tableau_r2 = pd.DataFrame(R2_MZ).T

print("\nComparison of McKelvey-Zavoina R² values:")
display(tableau_r2.round(4))

tableau_r2.to_csv(
    "comparaison_R2_McKelvey_Zavoina.csv"
)


Calculating R² values for model A...
Model A: Marginal R² = 0.281 [0.266 ; 0.295] | Conditional R² = 0.653 [0.648 ; 0.659]

Calculating R² values for model B...
Model B: Marginal R² = 0.127 [0.120 ; 0.134] | Conditional R² = 0.653 [0.647 ; 0.659]

Calculating R² values for model C...
Model C: Marginal R² = 0.130 [0.122 ; 0.137] | Conditional R² = 0.654 [0.647 ; 0.659]

Calculating R² values for model D...
Model D: Marginal R² = 0.137 [0.131 ; 0.145] | Conditional R² = 0.656 [0.650 ; 0.662]

Comparison of McKelvey-Zavoina R² values:


,R2_marginal_median,R2_marginal_IC_bas,R2_marginal_IC_haut,R2_conditionnel_median,R2_conditionnel_IC_bas,R2_conditionnel_IC_haut
A,0.2807,0.2659,0.2953,0.6534,0.6476,0.6592
B,0.1270,0.1198,0.1342,0.6528,0.6468,0.6587
C,0.1298,0.1225,0.1370,0.6535,0.6474,0.6594
D,0.1375,0.1305,0.1449,0.6563,0.6505,0.6619


In [20]:
# # ============================================================
# # 9. CLASSIFICATION ACCURACY (posterior predictive)
# # ============================================================

# def calcule_exactitude(
#     idata,
#     df_bayes,
#     predicteurs_modele,
#     col_voie="document_id",
#     taille_chunk=25
# ):
#     """
#     For each posterior draw:
#     - calculate the latent score eta (fixed effects + route effect);
#     - predict the class as the number of thresholds crossed by eta;
#     - return (exactitude_exacte, exactitude_a_une_classe_pres),
#       one pair per draw.

#     Requires y to be encoded as integers 0, 1, ..., K-1.
#     """

#     (
#         post, n_chains, n_draws,
#         seuils, betas, u, route_codes, X, y
#     ) = _extraire(
#         idata, df_bayes, predicteurs_modele, col_voie
#     )

#     n_sims = n_chains * n_draws

#     exact_draws  = np.empty(n_sims)
#     proche_draws = np.empty(n_sims)

#     y_col = y.reshape(-1, 1)

#     for d0 in range(0, n_sims, taille_chunk):

#         d1 = min(d0 + taille_chunk, n_sims)

#         # Latent score: predictors + route effect
#         eta = X @ betas[:, d0:d1]
#         eta += u[d0:d1, route_codes].T

#         # Predicted class = number of thresholds strictly crossed
#         # seuils is expected to have shape (n_sims, K-1); adapt if it has shape (K-1,)
#         seuils_chunk = (
#             seuils[d0:d1] if seuils.ndim == 2
#             else np.broadcast_to(seuils, (d1 - d0, seuils.shape[0]))
#         )

#         pred = (
#             eta[:, :, None] > seuils_chunk[None, :, :]
#         ).sum(axis=2).astype(np.int8)

#         exact_draws[d0:d1] = (pred == y_col).mean(axis=0)
#         proche_draws[d0:d1] = (
#             np.abs(pred - y_col) <= 1
#         ).mean(axis=0)

#     return exact_draws, proche_draws


# # ============================================================
# # APPLY TO THE FOUR MODELS
# # ============================================================

# EXACT = {}

# for nom in ["A", "B", "C", "D"]:

#     print(f"\nAccuracy for model {nom}...")

#     df_b, preds = PREPS[nom]

#     idata = az.from_netcdf(f"idata_{nom}.nc")

#     ex, pr = calcule_exactitude(
#         idata, df_b, preds
#     )

#     EXACT[nom] = {
#         "exact_median":  np.median(ex),
#         "exact_IC_bas":  np.quantile(ex, 0.025),
#         "exact_IC_haut": np.quantile(ex, 0.975),
#         "plus1_median":  np.median(pr),
#         "plus1_IC_bas":  np.quantile(pr, 0.025),
#         "plus1_IC_haut": np.quantile(pr, 0.975),
#     }

#     print(
#         f"Model {nom}: "
#         f"exact accuracy = {np.median(ex):.3f} "
#         f"[{np.quantile(ex, 0.025):.3f} ; "
#         f"{np.quantile(ex, 0.975):.3f}] | "
#         f"within ±1 class = {np.median(pr):.3f} "
#         f"[{np.quantile(pr, 0.025):.3f} ; "
#         f"{np.quantile(pr, 0.975):.3f}]"
#     )

#     del idata
#     gc.collect()


# # ============================================================
# # COMPARISON TABLE
# # ============================================================

# tableau_exact = pd.DataFrame(EXACT).T

# print("\nClassification accuracy:")
# display(tableau_exact.round(4))

# tableau_exact.to_csv(
#     "comparaison_exactitude.csv"
# )